In [ ]:
# Data science cw 2 
# 08/04/25 
# Faisa Hassan Sheikahmed

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score



## Salary Prediction Based on Job Attributes

### Hypothesis  1 - problem statement:
Salaries in data science roles are impacted by factors like company rating, job title, and location. This model aims to predict salary ranges based on these factors.

### Approach:
I used **linear regression** to predict salaries based on job attributes and evaluated the model’s performance using **5-fold cross-validation** to ensure generalization across different data splits. The Mean Squared Error (MSE) was used as the performance metric.


In [ ]:
# Load dataset into a pandas DataFrame
df = pd.read_csv("cleaned_dataset.csv")  

# Check for missing values and data types
df.head()  


In [ ]:
# Hypotehsis 1 Salary Prediction Based on Job Attributes 
# Linear regression

In [ ]:
# Encode Categorical Features

# One-hot encoding for categorical features
df_encoded = pd.get_dummies(df, drop_first=True)

# Define features and target variable
X = df_encoded.drop('avg_salary', axis=1)  # features
y = df_encoded['avg_salary']  # target variable (salary)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Create the Linear Regression Model
# Initialize the Linear Regression model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)



In [ ]:
## Make predictions on the test set
y_pred = model.predict(X_test)



In [ ]:
# evaluate model
# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate R-squared (R²) value
r2 = r2_score(y_test, y_pred)
print(f"R-squared: {r2}")


In [ ]:
import matplotlib.pyplot as plt

# Make the plot more user-friendly and informative with better spacing
plt.figure(figsize=(10, 6))  # Increase figure size for better clarity
plt.scatter(y_test, y_pred, alpha=0.6, edgecolors="w", s=100)  # Adjust alpha for transparency, edgecolors, and size for clarity
plt.xlabel('True Salary (USD)', fontsize=14)
plt.ylabel('Predicted Salary (USD)', fontsize=14)
plt.title('True vs Predicted Salary Comparison', fontsize=16)

# Add grid for better readability
plt.grid(True)

# Show the plot
plt.show()



In [ ]:
# Print the coefficients of the model
coefficients = pd.DataFrame(model.coef_, X.columns, columns=['Coefficient'])
print(coefficients)


5-Fold Cross-Validation to Evaluate Model Performance with MSE

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np

# Initialize the model
model = LinearRegression()

# Perform 5-fold cross-validation and compute negative MSE for each fold
cross_val_scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')

# Convert negative MSE to positive and print the average MSE across all folds
average_mse = -np.mean(cross_val_scores)
print(f"Average MSE from 5-fold cross-validation: {average_mse}")


## Regional Demand for Data Science Roles

### Hypothesis 2 - Problem Statement:
The demand for data scientists varies across different regions. Identifying areas with the highest demand helps companies direct their recruitment efforts and guides job seekers toward cities with the most opportunities.

### Approach:
I used **K-Means clustering**, an unsupervised learning technique, to group cities into clusters based on factors such as job availability, salary ranges, and regional demand for data science roles. This clustering model helps identify regions with the highest demand for data science talent. The results can inform recruitment strategies and guide job seekers toward cities with the most opportunities.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler


In [ ]:
# ENCODE CATERGORICAL DATA

# Encode 'Location' column
location_encoder = LabelEncoder()
df['Location_encoded'] = location_encoder.fit_transform(df['Location'])


In [ ]:
# FEAUTURE SELECTION

# Select features for clustering
features = ['Location_encoded', 'avg_salary', 'Rating']

# Extract the features
X = df[features]

In [ ]:
# STANDARIZE DATA

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# APPLY K MEANS CLUSTERING

# Apply K-Means clustering
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)  # You can experiment with different n_clusters
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Check the resulting clusters
df[['Location', 'Cluster', 'avg_salary', 'Rating']].head()


In [ ]:
# Analyze and Visualize Clusters

# Analyze the average salary and rating for each cluster
cluster_summary = df.groupby('Cluster')[['avg_salary', 'Rating']].mean()
print(cluster_summary)

# Visualize the clusters
plt.figure(figsize=(10, 6))
plt.scatter(df['avg_salary'], df['Rating'], c=df['Cluster'], cmap='viridis', alpha=0.7)
plt.xlabel('Average Salary (USD)', fontsize=14)
plt.ylabel('Rating', fontsize=14)
plt.title('Clusters of Data Science Roles by Salary and Rating', fontsize=16)
plt.show()



## Skillset Impact on Seniority and Salary
### Hypothesis 3 - Problem Statement:
Certain technical skills (e.g., Python) are correlated with higher seniority and better salary offers. Identifying these correlations helps job seekers focus on in-demand skills and enables employers to recruit more effectively.

### Approach:
I used K-Means clustering to group job roles based on skillsets, seniority, and salary levels. This clustering model helps identify which skills are linked to higher seniority and salary, providing valuable insights for both job seekers and employers. The results can inform recruitment strategies and guide job seekers in acquiring the most sought-after skills.

In [ ]:
# Select relevant columns for clustering
columns = ['python', 'excel', 'hadoop', 'spark', 'aws', 'tableau', 'big_data', 'Seniority', 'avg_salary']
df_selected = df[columns]

# Preview the data
df_selected.head()

In [ ]:
#Standardize the Data

from sklearn.preprocessing import StandardScaler

# Standardize the features
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_selected)

# Preview the standardized data
df_scaled[:5]


In [ ]:
# Apply K-Means Clustering

from sklearn.cluster import KMeans

# Apply K-Means clustering with 3 clusters
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(df_scaled)

# Check the resulting clusters
df[['JobTitle', 'Cluster', 'Seniority', 'avg_salary']].head()


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
# Analyze the Results

# Group by cluster and compute the average seniority and salary
cluster_summary = df.groupby('Cluster')[['Seniority', 'avg_salary']].mean()
print(cluster_summary)

# Apply PCA for dimensionality reduction (reduce to 2D)
pca = PCA(n_components=2)
df_pca = pca.fit_transform(df_scaled)

# Plot the clusters
plt.figure(figsize=(10, 6))
plt.scatter(df_pca[:, 0], df_pca[:, 1], c=df['Cluster'], cmap='viridis', alpha=0.7)
plt.xlabel('Principal Component 1', fontsize=14)
plt.ylabel('Principal Component 2', fontsize=14)
plt.title('Clustering of Job Roles by Skillset, Seniority, and Salary', fontsize=16)
plt.show()
